# 04 - Tracker Karşılaştırması

ByteTrack vs BoT-SORT performans karşılaştırması.

## Metrikler
- ID Switch sayısı
- Track tutarlılığı
- FPS etkisi
- Yoğun trafikte davranış

**ONEMLI:** Runtime > Change runtime type > T4 GPU secin!

In [ ]:
# Drive baglantisi + kurulum
import os
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/tez_models'
print(f"Drive OK: {os.listdir(SAVE_DIR)}")

!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from collections import defaultdict

# Fine-tuned model (Drive'dan)
MODEL_PATH = os.path.join(SAVE_DIR, 'istanbul_traffic_v1', 'weights', 'best.pt')
if os.path.exists(MODEL_PATH):
    print(f"Model OK: {MODEL_PATH}")
else:
    print("HATA: Model bulunamadi! Once 03 notebook'u calistirin.")
    print(f"Aranan: {MODEL_PATH}")
    print(f"Mevcut: {os.listdir(SAVE_DIR)}")

In [ ]:
# Test videolari (Drive'dan)
VIDEO_DIR = '/content/drive/MyDrive/İstanbul trafiği kayıt'

video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.avi', '.mov'))]
print(f"Drive'daki videolar: {video_files}")

# Ilk videoyu kullan (cam1)
VIDEO_PATH = os.path.join(VIDEO_DIR, video_files[0])
print(f"Kullanilacak video: {VIDEO_PATH}")

# Video bilgisi
cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f"Toplam kare: {total_frames}, FPS: {fps:.1f}")

NUM_FRAMES = min(500, total_frames)

TRACKERS = {
    'ByteTrack': 'bytetrack.yaml',
    'BoT-SORT': 'botsort.yaml',
}

In [ ]:
def evaluate_tracker(tracker_config, video_path, num_frames):
    """Tracker performansini olc."""
    model = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(video_path)

    frame_times = []
    all_track_ids = set()
    track_id_per_frame = []
    id_switches = 0
    prev_ids = set()

    for i in range(num_frames):
        ret, frame = cap.read()
        if not ret:
            break

        t0 = time.time()
        results = model.track(
            frame, conf=0.35, classes=[2, 3, 5, 7],
            tracker=tracker_config, persist=True, verbose=False
        )
        t1 = time.time()
        frame_times.append(t1 - t0)

        frame_ids = set()
        if results[0].boxes.id is not None:
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            frame_ids = set(ids)
            all_track_ids.update(frame_ids)

            # ID switch: onceki frame'de olan ama simdi olmayan ID'ler
            lost = prev_ids - frame_ids
            id_switches += len(lost)

        track_id_per_frame.append(frame_ids)
        prev_ids = frame_ids

    cap.release()

    return {
        'avg_fps': 1.0 / np.mean(frame_times),
        'total_unique_ids': len(all_track_ids),
        'avg_tracks_per_frame': np.mean([len(f) for f in track_id_per_frame]),
        'max_id': max(all_track_ids) if all_track_ids else 0,
        'id_switches': id_switches,
    }

results = {}
for name, config in TRACKERS.items():
    print(f'Degerlendiriliyor: {name}...')
    results[name] = evaluate_tracker(config, VIDEO_PATH, NUM_FRAMES)
    print(f'  FPS: {results[name]["avg_fps"]:.1f}, '
          f'Unique IDs: {results[name]["total_unique_ids"]}, '
          f'ID Switches: {results[name]["id_switches"]}')

In [ ]:
# Karsilastirma grafigi
import pandas as pd

df = pd.DataFrame(results).T
print(df)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

df['avg_fps'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Ortalama FPS')
axes[0].set_ylabel('FPS')

df['total_unique_ids'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Toplam Benzersiz Track ID')
axes[1].set_ylabel('ID Sayisi')

df['avg_tracks_per_frame'].plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Ort. Track / Kare')
axes[2].set_ylabel('Track Sayisi')

df['id_switches'].plot(kind='bar', ax=axes[3], color='tomato')
axes[3].set_title('ID Switch Sayisi')
axes[3].set_ylabel('Switch')

for ax in axes:
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()

# Drive'a kaydet
results_dir = os.path.join(SAVE_DIR, 'results')
os.makedirs(results_dir, exist_ok=True)
plt.savefig(os.path.join(results_dir, 'tracker_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nGrafik kaydedildi: {results_dir}/tracker_comparison.png")